# RNN Language Model Exploration

This notebook explores RNN language models with PyTorch and the WikiText dataset.

## Table of Contents
1. [Setup and Imports](#Setup-and-Imports)
2. [Data Exploration](#Data-Exploration)
3. [Model Architecture](#Model-Architecture)
4. [Training Process](#Training-Process)
5. [Text Generation](#Text-Generation)
6. [Autograd Demonstration](#Autograd-Demonstration)
7. [Experiments](#Experiments)

## Setup and Imports

In [ ]:
# Add project root to Python path
import sys
import os
project_root = os.path.dirname(os.getcwd())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Standard imports
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import time
from datetime import datetime

# Project imports
from models.rnn_model import RNNLanguageModel, count_parameters
from src.data_preprocessing import preprocess_wikitext, Vocabulary
from src.generate import generate_text

# Set style
plt.style.use('default')
sns.set_palette('husl')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name()}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Data Exploration

In [ ]:
# Load and explore the WikiText dataset
print("Loading WikiText dataset...")

try:
    data_loaders, vocab = preprocess_wikitext(
        dataset_name="wikitext-2-v1",
        vocab_size=5000,
        seq_length=64,
        batch_size=16,
        cache_dir="../data"
    )
    
    print(f"✓ Dataset loaded successfully!")
    print(f"Vocabulary size: {len(vocab):,}")
    
    for split, loader in data_loaders.items():
        print(f"{split.capitalize()}: {len(loader)} batches")
        
except Exception as e:
    print(f"Error loading dataset: {e}")
    print("This is likely due to missing dependencies. Please install the requirements.")

In [ ]:
# Explore vocabulary statistics
if 'vocab' in locals():
    # Most common tokens
    most_common = vocab.token_counts.most_common(20)
    
    tokens, counts = zip(*most_common)
    
    plt.figure(figsize=(12, 6))
    plt.bar(range(len(tokens)), counts)
    plt.xlabel('Token Rank')
    plt.ylabel('Frequency')
    plt.title('Top 20 Most Frequent Tokens')
    plt.xticks(range(len(tokens)), tokens, rotation=45)
    plt.tight_layout()
    plt.show()
    
    # Token frequency distribution
    freq_counts = Counter(vocab.token_counts.values())
    frequencies = list(freq_counts.keys())
    freq_counts_values = list(freq_counts.values())
    
    plt.figure(figsize=(10, 6))
    plt.loglog(frequencies, freq_counts_values, 'o-')
    plt.xlabel('Token Frequency')
    plt.ylabel('Number of Tokens')
    plt.title('Token Frequency Distribution (Zipf\'s Law)')
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
# Examine sample data
if 'data_loaders' in locals():
    train_loader = data_loaders['train']
    sample_batch = next(iter(train_loader))
    input_seq, target_seq = sample_batch
    
    print(f"Batch shapes:")
    print(f"Input: {input_seq.shape}")
    print(f"Target: {target_seq.shape}")
    
    # Decode and show sample sequences
    print(f"\nSample sequences:")
    for i in range(min(3, input_seq.size(0))):
        input_text = vocab.decode(input_seq[i].tolist())
        target_text = vocab.decode(target_seq[i].tolist())
        
        print(f"\nSequence {i+1}:")
        print(f"Input:  {input_text[:100]}...")
        print(f"Target: {target_text[:100]}...")

## Model Architecture

In [ ]:
# Create and analyze different model architectures
if 'vocab' in locals():
    vocab_size = len(vocab)
else:
    vocab_size = 10000  # Default for demo

# Model configurations
configs = [
    {'name': 'Small LSTM', 'type': 'LSTM', 'embed': 128, 'hidden': 256, 'layers': 1},
    {'name': 'Medium LSTM', 'type': 'LSTM', 'embed': 256, 'hidden': 512, 'layers': 2},
    {'name': 'Large LSTM', 'type': 'LSTM', 'embed': 512, 'hidden': 1024, 'layers': 3},
    {'name': 'Medium GRU', 'type': 'GRU', 'embed': 256, 'hidden': 512, 'layers': 2},
]

models_info = []

for config in configs:
    model = RNNLanguageModel(
        vocab_size=vocab_size,
        embed_size=config['embed'],
        hidden_size=config['hidden'],
        num_layers=config['layers'],
        rnn_type=config['type']
    )
    
    param_count = count_parameters(model)
    
    models_info.append({
        'name': config['name'],
        'type': config['type'],
        'parameters': param_count,
        'embed_size': config['embed'],
        'hidden_size': config['hidden'],
        'num_layers': config['layers']
    })
    
    print(f"{config['name']}:")
    print(f"  Parameters: {param_count:,}")
    print(f"  Embedding: {config['embed']} → Hidden: {config['hidden']} → Vocab: {vocab_size}")
    print()

In [ ]:
# Visualize model complexity
if models_info:
    names = [info['name'] for info in models_info]
    params = [info['parameters'] for info in models_info]
    
    plt.figure(figsize=(10, 6))
    bars = plt.bar(names, params)
    plt.xlabel('Model Configuration')
    plt.ylabel('Number of Parameters')
    plt.title('Model Complexity Comparison')
    plt.yscale('log')
    
    # Add value labels on bars
    for bar, param in zip(bars, params):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height()*1.1, 
                f'{param:,}', ha='center', va='bottom', rotation=0)
    
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## Training Process

In [ ]:
# Demonstrate training process with a small model
if 'data_loaders' in locals() and 'vocab' in locals():
    # Create a small model for quick training demo
    model = RNNLanguageModel(
        vocab_size=len(vocab),
        embed_size=128,
        hidden_size=256,
        num_layers=1,
        rnn_type='LSTM',
        dropout=0.2
    )
    
    model = model.to(device)
    print(f"Training model with {count_parameters(model):,} parameters")
    
    # Training setup
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    
    # Training loop (just a few batches for demo)
    model.train()
    train_losses = []
    
    print("\nTraining for 10 steps...")
    start_time = time.time()
    
    for step, (input_seq, target_seq) in enumerate(data_loaders['train']):
        if step >= 10:  # Only train for 10 steps
            break
            
        input_seq, target_seq = input_seq.to(device), target_seq.to(device)
        batch_size = input_seq.size(0)
        
        # Forward pass
        optimizer.zero_grad()
        hidden = model.init_hidden(batch_size, device)
        output, hidden = model(input_seq, hidden)
        
        # Loss calculation
        loss = criterion(output.view(-1, output.size(-1)), target_seq.view(-1))
        
        # Backward pass
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        
        train_losses.append(loss.item())
        
        if step % 2 == 0:
            print(f"Step {step+1}/10: Loss = {loss.item():.4f}")
    
    elapsed = time.time() - start_time
    print(f"\nCompleted in {elapsed:.2f} seconds")
    
    # Plot training loss
    plt.figure(figsize=(8, 5))
    plt.plot(train_losses, 'b-o')
    plt.xlabel('Training Step')
    plt.ylabel('Loss')
    plt.title('Training Loss (Demo - 10 steps)')
    plt.grid(True, alpha=0.3)
    plt.show()
    
else:
    print("Data loaders not available. Skipping training demo.")

## Text Generation

In [ ]:
# Text generation experiments
if 'model' in locals() and 'vocab' in locals():
    model.eval()
    
    # Test different generation strategies
    prompts = [
        "The weather today",
        "Machine learning is",
        "In the future"
    ]
    
    strategies = [
        {'name': 'Conservative', 'temp': 0.5, 'top_k': 10},
        {'name': 'Balanced', 'temp': 0.8, 'top_k': 40},
        {'name': 'Creative', 'temp': 1.2, 'top_k': 100}
    ]
    
    print("Text Generation Examples:")
    print("=" * 50)
    
    for prompt in prompts:
        print(f"\nPrompt: '{prompt}'")
        print("-" * 30)
        
        for strategy in strategies:
            try:
                generated = generate_text(
                    model=model,
                    vocab=vocab,
                    prompt=prompt,
                    max_length=30,
                    temperature=strategy['temp'],
                    top_k=strategy['top_k'],
                    device=device
                )
                
                print(f"{strategy['name']:12s}: {generated}")
                
            except Exception as e:
                print(f"{strategy['name']:12s}: Error - {e}")
    
else:
    print("Model not available. Skipping text generation.")

## Autograd Demonstration

In [ ]:
# Demonstrate PyTorch's automatic differentiation
print("PyTorch Autograd Demonstration")
print("=" * 40)

# Simple example: y = x^2 + 2x + 1
print("\n1. Simple function: y = x² + 2x + 1")

x = torch.tensor([2.0], requires_grad=True)
print(f"x = {x.item()}")

y = x**2 + 2*x + 1
print(f"y = x² + 2x + 1 = {y.item()}")

# Compute gradient: dy/dx = 2x + 2
y.backward()
print(f"dy/dx = 2x + 2 = {x.grad.item()} (expected: {2*x.item() + 2})")

In [ ]:
# Neural network autograd example
print("\n2. Neural Network Autograd Example")

# Create simple network
vocab_size_demo = 100
embed_dim = 16
hidden_dim = 32

# Layers
embedding = nn.Embedding(vocab_size_demo, embed_dim)
linear1 = nn.Linear(embed_dim, hidden_dim)
linear2 = nn.Linear(hidden_dim, vocab_size_demo)

# Input data
input_ids = torch.randint(0, vocab_size_demo, (2, 5))
targets = torch.randint(0, vocab_size_demo, (2, 5))

print(f"Input shape: {input_ids.shape}")
print(f"Target shape: {targets.shape}")

# Forward pass
embedded = embedding(input_ids)
hidden = torch.relu(linear1(embedded))
logits = linear2(hidden)

print(f"Embedded shape: {embedded.shape}")
print(f"Hidden shape: {hidden.shape}")
print(f"Logits shape: {logits.shape}")

# Compute loss
criterion = nn.CrossEntropyLoss()
loss = criterion(logits.view(-1, vocab_size_demo), targets.view(-1))

print(f"Loss: {loss.item():.4f}")

# Backward pass
loss.backward()

# Show gradients
print(f"\nGradient norms:")
print(f"Embedding: {embedding.weight.grad.norm():.4f}")
print(f"Linear1: {linear1.weight.grad.norm():.4f}")
print(f"Linear2: {linear2.weight.grad.norm():.4f}")

# Visualize computational graph concept
print(f"\nComputational Graph Flow:")
print(f"input_ids → embedding → linear1 → relu → linear2 → loss")
print(f"        ← gradients ← gradients ← gradients ← gradients ←")

## Experiments

In [ ]:
# Experiment: Effect of temperature on text generation
if 'model' in locals() and 'vocab' in locals():
    print("Experiment: Effect of Temperature on Text Generation")
    print("=" * 55)
    
    prompt = "The future of artificial intelligence"
    temperatures = [0.1, 0.5, 0.8, 1.0, 1.5, 2.0]
    
    print(f"Prompt: '{prompt}'\n")
    
    for temp in temperatures:
        try:
            generated = generate_text(
                model=model,
                vocab=vocab,
                prompt=prompt,
                max_length=25,
                temperature=temp,
                top_k=50,
                device=device
            )
            
            print(f"T={temp:3.1f}: {generated}")
            
        except Exception as e:
            print(f"T={temp:3.1f}: Error - {e}")
    
    print("\nObservations:")
    print("- Lower temperature (T < 1.0): More conservative, repetitive")
    print("- Higher temperature (T > 1.0): More creative, potentially incoherent")
    print("- T ≈ 0.8-1.0: Good balance between creativity and coherence")

else:
    print("Model not available for experiments.")

In [ ]:
# Experiment: Sequence length analysis
if 'vocab' in locals():
    print("\nExperiment: Analyzing different sequence lengths")
    print("=" * 50)
    
    sequence_lengths = [16, 32, 64, 128, 256]
    batch_size = 8
    
    timing_results = []
    memory_results = []
    
    for seq_len in sequence_lengths:
        # Create model for this sequence length
        test_model = RNNLanguageModel(
            vocab_size=len(vocab) if 'vocab' in locals() else 1000,
            embed_size=128,
            hidden_size=256,
            num_layers=1,
            rnn_type='LSTM'
        ).to(device)
        
        # Create dummy input
        dummy_input = torch.randint(0, len(vocab) if 'vocab' in locals() else 1000, 
                                   (batch_size, seq_len)).to(device)
        
        # Time forward pass
        torch.cuda.synchronize() if device.type == 'cuda' else None
        start_time = time.time()
        
        with torch.no_grad():
            hidden = test_model.init_hidden(batch_size, device)
            output, _ = test_model(dummy_input, hidden)
        
        torch.cuda.synchronize() if device.type == 'cuda' else None
        elapsed = time.time() - start_time
        
        timing_results.append(elapsed * 1000)  # Convert to ms
        
        # Memory usage (approximate)
        if device.type == 'cuda':
            memory_mb = torch.cuda.max_memory_allocated() / 1024**2
            torch.cuda.reset_peak_memory_stats()
        else:
            memory_mb = 0  # Not available for CPU
        
        memory_results.append(memory_mb)
        
        print(f"Seq len {seq_len:3d}: {elapsed*1000:5.1f}ms", end="")
        if device.type == 'cuda':
            print(f", {memory_mb:.1f}MB")
        else:
            print()
    
    # Plot results
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # Timing plot
    ax1.plot(sequence_lengths, timing_results, 'bo-')
    ax1.set_xlabel('Sequence Length')
    ax1.set_ylabel('Forward Pass Time (ms)')
    ax1.set_title('Timing vs Sequence Length')
    ax1.grid(True, alpha=0.3)
    
    # Memory plot
    if device.type == 'cuda':
        ax2.plot(sequence_lengths, memory_results, 'ro-')
        ax2.set_xlabel('Sequence Length')
        ax2.set_ylabel('Memory Usage (MB)')
        ax2.set_title('Memory vs Sequence Length')
        ax2.grid(True, alpha=0.3)
    else:
        ax2.text(0.5, 0.5, 'Memory tracking\nnot available\nfor CPU', 
                ha='center', va='center', transform=ax2.transAxes)
        ax2.set_title('Memory Usage (GPU only)')
    
    plt.tight_layout()
    plt.show()

else:
    print("Vocabulary not available for sequence length analysis.")

## Conclusion

This notebook demonstrated:

1. **Data Processing**: How to load and preprocess WikiText dataset
2. **Model Architecture**: Different RNN configurations and their complexity
3. **Training Process**: Basic training loop with loss tracking
4. **Text Generation**: Various generation strategies with temperature and top-k
5. **Autograd**: PyTorch's automatic differentiation system
6. **Experiments**: Effect of hyperparameters on model behavior

### Next Steps:

- Train full models with more epochs
- Experiment with different architectures (GRU, attention mechanisms)
- Compare with transformer models
- Deploy models via the API for interactive use

### Key Takeaways:

- RNNs are effective for sequence modeling but suffer from vanishing gradients
- LSTM/GRU variants address some RNN limitations
- Temperature controls creativity vs coherence in text generation
- PyTorch's autograd makes gradient computation automatic and efficient

In [ ]:
# Save any trained models for later use
if 'model' in locals():
    checkpoint_path = '../checkpoints/notebook_model.pth'
    os.makedirs('../checkpoints', exist_ok=True)
    
    checkpoint = {
        'model_state_dict': model.state_dict(),
        'model_config': {
            'vocab_size': model.vocab_size,
            'embed_size': model.embed_size,
            'hidden_size': model.hidden_size,
            'num_layers': model.num_layers,
            'model_type': model.rnn_type.lower()
        },
        'training_info': 'Demo training from notebook'
    }
    
    torch.save(checkpoint, checkpoint_path)
    print(f"Model checkpoint saved to: {checkpoint_path}")
    
    # Also save vocabulary if available
    if 'vocab' in locals():
        vocab_path = '../data/notebook_vocab.pkl'
        vocab.save(vocab_path)
        print(f"Vocabulary saved to: {vocab_path}")

print(f"\nNotebook completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")